# He Thong Dich Tieu Thuyet AI: Qwen3-4B (Unsloth QLoRA 4-bit + SQL Server RAG)

Notebook nay bao gom toan bo quy trinh tren **Google Colab (GPU T4)** cho 2 giai doan:
- **Giai doan 2 (Fine-tune Model):** Doc file `data_train.jsonl` xuat tu bang `dbo.TranslationExamples` va `dbo.PronounRules`, huan luyen QLoRA 4-bit tren `Qwen3-4B-Instruct` va luu bo trong so vao `/content/drive/MyDrive/NovelAI/lora_adapter`.
- **Giai doan 3 (Translate New Chapter - SQL RAG + QLoRA):** Nap `lora_adapter` tu Google Drive, doc file `pending_chunks.json` chua ngu canh RAG, dich tu dong tung doan kem co che Checkpoint Resume va dem `input_tokens` / `output_tokens` ghi ra `translated_chunks.json` de cap nhat ve bang `dbo.TranslationChunks`.

### Cau truc thu muc tren Google Drive:
- `/MyDrive/NovelAI/dataset/data_train.jsonl`: Du lieu huan luyen Giai doan 2
- `/MyDrive/NovelAI/lora_adapter/`: Noi luu LoRA Adapter (~100MB) sau khi Fine-tune
- `/MyDrive/NovelAI/input/pending_chunks.json`: Cac doan chuong moi cho dich kem SQL RAG
- `/MyDrive/NovelAI/output/translated_chunks.json`: Ket qua dich va thong ke Token tra ve SQL Server

## Buoc 1: Cai dat thu vien Unsloth & Ket noi Google Drive
Chay o ben duoi dau tien trong moi phien lam viec (chon **Runtime -> Change runtime type -> T4 GPU**).

In [ ]:
!pip install -q unsloth

import json
import os
import torch
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Khoi tao cay thu muc chuan tren Google Drive
BASE_DRIVE_DIR = "/content/drive/MyDrive/NovelAI"
DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "dataset")
LORA_DIR = os.path.join(BASE_DRIVE_DIR, "lora_adapter")
INPUT_DIR = os.path.join(BASE_DRIVE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DRIVE_DIR, "output")

for folder in [DATASET_DIR, LORA_DIR, INPUT_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

TRAIN_DATA_PATH = os.path.join(DATASET_DIR, "data_train.jsonl")
PENDING_CHUNKS_PATH = os.path.join(INPUT_DIR, "pending_chunks.json")
TRANSLATED_CHUNKS_PATH = os.path.join(OUTPUT_DIR, "translated_chunks.json")

print("Da san sang cay thu muc tai:", BASE_DRIVE_DIR)
print("GPU hien tai:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Chua bat GPU!")

### (Tuy chon) Tao file du lieu mau de chay thu ngay tren Colab
Neu ban chua kip xuat file `data_train.jsonl` hoac `pending_chunks.json` tu SQL Server len Google Drive, o code nay se tao san du lieu mau khop voi bo truyen *Blue Archive* neu file chua ton tai.

In [ ]:
if not os.path.exists(TRAIN_DATA_PATH):
    sample_train_data = [
        {
            "instruction": "Dich truyen tieng Anh sang tieng Viet",
            "context": {
                "speaker": "Yuuka",
                "listener": "Sensei",
                "scene": "hostile",
                "pronouns": {"I": "ta / toi", "you": "nguoi"}
            },
            "input": "\"Why are you alive? Could it be, you weren't there?\"",
            "output": "\"Tai sao nguoi con song? Chang le nguoi khong he o do sao?\""
        },
        {
            "instruction": "Dich truyen tieng Anh sang tieng Viet",
            "context": {
                "speaker": "Sensei",
                "listener": "Yuuka",
                "scene": "worried",
                "pronouns": {"I": "thay", "you": "em"}
            },
            "input": "\"Yuuka! Thank goodness. Are you safe? Can you tell me what's going on right now?\"",
            "output": "\"Yuuka! Cam on Chua. Em co an toan khong? Em co the noi cho thay biet chuyen gi dang xay ra duoc khong?\""
        },
        {
            "instruction": "Dich truyen tieng Anh sang tieng Viet",
            "context": {
                "speaker": "Black Suit",
                "listener": "Sensei",
                "scene": "warning",
                "pronouns": {"I": "toi", "you": "anh"}
            },
            "input": "\"From now on, you must not trust any of the students. If you want to live, you must run away.\"",
            "output": "\"Tu gio tro di, anh khong duoc tin tuong bat ky hoc sinh nao. Neu muon song, anh phai chay tron.\""
        }
    ]
    with open(TRAIN_DATA_PATH, "w", encoding="utf-8") as f:
        for item in sample_train_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("Da tao file mau data_train.jsonl tai:", TRAIN_DATA_PATH)
else:
    print("Da tim thay file data_train.jsonl tren Google Drive:", TRAIN_DATA_PATH)

if not os.path.exists(PENDING_CHUNKS_PATH):
    sample_pending = [
        {
            "chunk_id": 1,
            "chapter_id": 1,
            "chapter_number": 3,
            "chunk_index": 1,
            "source_hash": "demo_source_hash_001",
            "context_hash": "demo_context_hash_001",
            "instruction": "Dich truyen tieng Anh sang tieng Viet",
            "context": {
                "style_guide": "Ngoi ke thu 3 goi Sensei la 'anh'. Doc thoai noi tam de trong dau '...' va xung 'minh'. Thong bao de trong dau 「...」.",
                "pronoun_rules": [
                    {"speaker": "Sensei", "listener": "Arona/Plana", "scene": "urgent", "pronouns": {"I": "thay", "you": "hai em"}},
                    {"speaker": "Plana", "listener": "Sensei", "scene": "urgent", "pronouns": {"I": "em", "you": "Sensei / thay"}}
                ],
                "few_shot_examples": [
                    {
                        "speaker": "Plana",
                        "listener": "Sensei",
                        "scene": "urgent",
                        "input": "\"Sensei, you have to run away right now. You have to get a hold of yourself!\"",
                        "output": "\"Sensei, thay phai chay ngay di. Thay phai giu binh tinh!\""
                    }
                ]
            },
            "input": "In the taxi going back the way they came, Sensei was sweating bullets.\n'Why am I suddenly being attacked?'\n\"Sensei, this is not looking good. We need to go to another area.\""
        }
    ]
    with open(PENDING_CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(sample_pending, f, ensure_ascii=False, indent=2)
    print("Da tao file mau pending_chunks.json tai:", PENDING_CHUNKS_PATH)
else:
    print("Da tim thay file pending_chunks.json tren Google Drive:", PENDING_CHUNKS_PATH)

---
# GIAI DOAN 2: FINE-TUNE QWEN3-4B VOI UNSLOTH QLORA 4-BIT
*Chi chay Giai doan 2 khi ban muon huan luyen model hoc van phong va quy tac xung ho tu `data_train.jsonl`. Neu ban da co san thu muc `lora_adapter` tren Drive va chi muon dich chuong moi, hay cuon thang xuong **Giai doan 3**.*

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
BASE_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

# 1. Tai Base Model 4-bit va Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

# 2. Gan cau hinh QLoRA vao 7 lop Attention & MLP
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Da khoi tao Qwen3-4B QLoRA 4-bit thanh cong!")

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

SYSTEM_PROMPT = (
    "Ban la dich gia tieu thuyet chuyen nghiep. Hay dich van ban tieng Anh o phan 'input' sang tieng Viet muot ma, "
    "giu nguyen dau '...' cho doc thoai noi tam, dau 「...」 cho thong bao va tuan thu tuyet doi quy tac xung ho trong 'context'. "
    "Chi xuat ra ban dich tieng Viet, khong giai thich gi them."
)

# 1. Nap tap du lieu data_train.jsonl tu Google Drive
raw_dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train")

def format_json_to_chat(example):
    ctx_str = json.dumps(example["context"], ensure_ascii=False, indent=2)
    user_content = (
        f"### Nhiem vu:\n{example.get('instruction', 'Dich truyen tieng Anh sang tieng Viet')}\n\n"
        f"### Ngu canh & Quy tac xung ho (context):\n{ctx_str}\n\n"
        f"### Van ban tieng Anh can dich (input):\n{example['input']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False
    )
    return {"text": text}

train_dataset = raw_dataset.map(format_json_to_chat)
print(f"Da nap {len(train_dataset)} mau huan luyen tu {TRAIN_DATA_PATH}")

# 2. Cau hinh SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Chi tinh Loss tren phan ban dich tieng Viet cua Assistant
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# 3. Chay Fine-tune
trainer_stats = trainer.train()

# 4. Luu LoRA Adapter (~100MB) vao Google Drive
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("Da luu thanh cong LoRA Adapter vao Google Drive tai:", LORA_DIR)

### Buoc kiem thu nhanh (Sanity Check) ngay sau khi Fine-tune

In [ ]:
FastLanguageModel.for_inference(model)

test_sample = {
    "instruction": "Dich truyen tieng Anh sang tieng Viet",
    "context": {
        "speaker": "Yuuka",
        "listener": "Sensei",
        "scene": "hostile",
        "pronouns": {"you": "nguoi"}
    },
    "input": "\"Why are you alive? Could it be, you weren't there?\""
}

user_test = (
    f"### Nhiem vu:\n{test_sample['instruction']}\n\n"
    f"### Ngu canh & Quy tac xung ho (context):\n{json.dumps(test_sample['context'], ensure_ascii=False, indent=2)}\n\n"
    f"### Van ban tieng Anh can dich (input):\n{test_sample['input']}"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_test}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.2, use_cache=True)

result_vi = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("Cau goc EN:", test_sample["input"])
print("Ban dich VI:", result_vi)

---
# GIAI DOAN 3: DICH CHUONG MOI KET HOP SQL RAG + QLORA (BATCH INFERENCE)
*O code duoi day co the chay doc lap o bat ky phien Colab moi nao (chi can chay **Buoc 1** roi chay thang o nay). He thong se tu dong nap `lora_adapter` tu Google Drive, doc `pending_chunks.json`, dich tung doan kem Checkpoint Resume va xuat `translated_chunks.json`.*

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
BASE_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

# 1. Kiem tra neu model chua co trong RAM thi nap tu Google Drive (hoac Base Model neu chua train)
if "model" not in globals():
    has_lora = os.path.exists(os.path.join(LORA_DIR, "adapter_config.json"))
    active_model_path = LORA_DIR if has_lora else BASE_MODEL_NAME
    model_label = "Qwen3-4B-QLoRA" if has_lora else "Qwen3-4B-Base-RAG"
    print(f"Dang tai mo hinh tu: {active_model_path} ({model_label})")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = active_model_path,
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = True,
    )
else:
    model_label = "Qwen3-4B-QLoRA"
    print("Su dung mo hinh dang co san trong VRAM GPU.")

FastLanguageModel.for_inference(model)

# 2. Doc danh sach cac doan cho dich tu pending_chunks.json
with open(PENDING_CHUNKS_PATH, "r", encoding="utf-8") as f:
    pending_chunks = json.load(f)

# 3. Doc Checkpoint neu da dich do dang truoc do
completed_map = {}
if os.path.exists(TRANSLATED_CHUNKS_PATH):
    with open(TRANSLATED_CHUNKS_PATH, "r", encoding="utf-8") as f:
        for item in json.load(f):
            completed_map[item["chunk_id"]] = item

INFERENCE_SYSTEM_PROMPT = (
    "Ban la dich gia tieu thuyet chuyen nghiep. Hay dich doan van tieng Anh o phan 'input' sang tieng Viet muot ma, "
    "tuan thu nghiem ngat quy tac xung ho trong 'pronoun_rules' va van phong trong 'few_shot_examples'. "
    "Chi xuat ra ban dich tieng Viet, tuyet doi khong giai thich hay them ghi chu."
)

prev_translated_tail = ""
last_chapter_id = None

for item in pending_chunks:
    cid = item["chunk_id"]
    if cid in completed_map:
        prev_translated_tail = completed_map[cid]["output"][-250:]
        last_chapter_id = item.get("chapter_id")
        print(f"[Skip] ChunkId={cid} da duoc dich truoc do.")
        continue

    if item.get("chapter_id") != last_chapter_id:
        prev_translated_tail = ""
        last_chapter_id = item.get("chapter_id")

    ctx = item.get("context", {})
    style_guide = ctx.get("style_guide", "")
    pronoun_rules = json.dumps(ctx.get("pronoun_rules", []), ensure_ascii=False, indent=2)
    few_shot = json.dumps(ctx.get("few_shot_examples", []), ensure_ascii=False, indent=2)

    user_prompt = (
        f"### Huong dan van phong:\n{style_guide}\n\n"
        f"### Ngu canh & Quy tac xung ho (SQL RAG):\n{pronoun_rules}\n\n"
        f"### Cau dich mau da duyet:\n{few_shot}\n\n"
    )
    if prev_translated_tail:
        user_prompt += f"### Doan dich ngay truoc do (Chi doc de noi mach van, KHONG dich lai):\n{prev_translated_tail}\n\n"
    user_prompt += f"### Van ban tieng Anh can dich (input):\n{item['input']}"

    messages = [
        {"role": "system", "content": INFERENCE_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True
    )
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    input_token_count = int(inputs.input_ids.shape[1])

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1500,
        temperature = 0.3,
        top_p = 0.9,
        use_cache = True
    )

    generated_ids = outputs[0][input_token_count:]
    output_token_count = int(len(generated_ids))
    vi_translation = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    completed_map[cid] = {
        "chunk_id": cid,
        "chapter_id": item.get("chapter_id"),
        "chapter_number": item.get("chapter_number"),
        "chunk_index": item.get("chunk_index"),
        "model_name": model_label,
        "input_tokens": input_token_count,
        "output_tokens": output_token_count,
        "output": vi_translation
    }

    with open(TRANSLATED_CHUNKS_PATH, "w", encoding="utf-8") as out_f:
        json.dump(list(completed_map.values()), out_f, ensure_ascii=False, indent=2)

    prev_translated_tail = vi_translation[-250:]
    print(f"Hoan tat ChunkId={cid} (Doan {item.get('chunk_index')}) | Tokens: In={input_token_count}, Out={output_token_count}")

print("\nDa luu toan bo ket qua dich tai:", TRANSLATED_CHUNKS_PATH)